In [7]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse

URL = "https://www.healthline.com/nutrition/best-foods-for-kidneys"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

print("Status:", response.status_code)
print("Final URL:", response.url)
print("HTML size:", len(response.text))

Status: 200
Final URL: https://www.healthline.com/nutrition/best-foods-for-kidneys
HTML size: 650497


In [8]:
def text_of(tag):
    return " ".join(tag.get_text(" ", strip=True).split())

all_links = soup.find_all("a", href=True)
link_labels = [text_of(a) for a in all_links]

start = None
for i, label in enumerate(link_labels):
    if label.strip().casefold() == "foods to help kidneys":
        start = i
        break

if start is None:
    print("Page title:", soup.title.get_text(" ", strip=True) if soup.title else "No title")
    print("Links containing 'kidney':", [
        label for label in link_labels if "kidney" in label.casefold()
    ][:30])
    raise RuntimeError(
        "The 'Foods to help kidneys' TOC link was not found. "
        "The server may have returned a block page or Healthline may have changed its markup."
    )

food_items = []
for a in all_links[start + 1:]:
    label = text_of(a)
    normalized = label.casefold().strip()

    if normalized == "faq" or normalized.startswith("frequently asked"):
        break

    href = a.get("href", "")
    if href.startswith("#") and label:
        food_items.append(label)

food_items = list(dict.fromkeys(food_items))

print("Food items extracted from TOC:", len(food_items))
for i, name in enumerate(food_items, 1):
    print(f"{i}. {name}")

if len(food_items) != 20:
    raise RuntimeError(
        f"Expected 20 TOC food links, found {len(food_items)}. "
        "No manual fallback is used. Inspect the links printed above and the downloaded HTML."
    )

Food items extracted from TOC: 20
1. Cauliflower
2. Blueberries
3. Sea bass
4. Red grapes
5. Egg white
6. Garlic
7. Buckwheat
8. Olive oil
9. Bulgur
10. Cabbage
11. Chicken
12. Peppers
13. Onions
14. Arugula
15. Macadamia nuts
16. Radish
17. Turnips
18. Pineapple
19. Cranberries
20. Mushrooms


In [10]:
df = pd.DataFrame({"food_item": food_items})

output_file = "kidney_friendly_foods.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} food items to {output_file}")
display(df)

Saved 20 food items to kidney_friendly_foods.csv


,food_item
0,Cauliflower
1,Blueberries
2,Sea bass
3,Red grapes
4,Egg white
5,Garlic
6,Buckwheat
7,Olive oil
8,Bulgur
9,Cabbage
